# 2 提示词基线

对应文档 2.2 – 2.4。三种提示词各跑一遍 GSM8K，逐条打分，统计三类输出。

这一步不训练，整条链路是直的：**读数据 → 套模板 → 发请求 → 打分 → 统计**。
得到的分数就是第 3 节 GRPO 要对比的基线。

> 完整可运行的脚本在 `scripts/prompting_baselines.py`，这里只把关键环节拆开看。

In [ ]:
import os
import sys

# 仓库根目录，后面所有相对路径都相对它
os.chdir(r"C:\Users\intangible\Desktop\easy-llm-alignment")

# 公共模块在 utils/ 下，加进搜索路径才 import 得到
sys.path.insert(0, os.getcwd())

print("工作目录:", os.getcwd())
# 工作目录: C:\Users\intangible\Desktop\easy-llm-alignment

## 2.2 三种提示词

三份模板的差别只在结尾那一段：怎么让模型把答案交出来。

### question_only

问题后面直接跟一句格式要求，没有推理引导。

```text
{question} Please put your final answer within \\boxed{{}}.
```

### r1_zero

先用一段 system 说明要求模型先想再答，并用两个标签把思考过程和答案分开；最后给出 `<think>` 开标签，让模型接着往下写。

```text
A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>
```

### r1_zero_three_shot

在 r1_zero 前面塞三组完整问答，结构与真题完全一样，真题换到最后一组。下面是截断版，长行用 `...` 省略。

```text
A conversation between User and Assistant. The User asks a question, and the Assistant solves it. ...
User: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
Assistant: <think> There are 15 trees originally. ... </think> <answer> 6 </answer>
User: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
Assistant: <think> ... 3 + 2 = 5. </think> <answer> 5 </answer>
User: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
Assistant: <think> ... 74 - 35 = 39. </think> <answer> 39 </answer>
User: {question}
Assistant: <think>
```

`question_only.prompt` 里写的是 `\\boxed{{}}`。`format` 只处理花括号（`{{}}` → `{}`），两个反斜杠不动，所以模型收到的是 `\\boxed{}`。官方仓库里就是这个写法。

In [ ]:
import json
from pathlib import Path

first = json.loads(open("data/gsm8k/test.jsonl", encoding="utf-8").readline())
question = first["question"]


def build_prompt(template_name, question):
    '''
    读模板并把 {question} 换成题目

    template_name: str  prompts/ 下的文件名
    question: str       题目原文
    '''
    template = Path("prompts", template_name).read_text(encoding="utf-8")
    return template.format(question=question)


def show(prompt, head=3, tail=2):
    '''太长的只打印首尾。每行都超过 100 字符，全打出来会糊成一片'''
    lines = prompt.splitlines()
    if len(lines) <= head + tail:
        print(prompt)
    else:
        print("\n".join(lines[:head]))
        print(f"... 省略 {len(lines) - head - tail} 行 ...")
        print("\n".join(lines[-tail:]))


for name in ["question_only.prompt", "r1_zero.prompt", "r1_zero_three_shot_gsm8k.prompt"]:
    prompt = build_prompt(name, question)
    print(f"===== {name}（{len(prompt)} 字符 / {prompt.count(chr(10)) + 1} 行）")
    show(prompt)
    print()

# ===== question_only.prompt（327 字符 / 1 行）
# Janet's ducks lay 16 eggs per day. ... Please put your final answer within \\boxed{}.
#
# ===== r1_zero.prompt（719 字符 / 3 行）
# A conversation between User and Assistant. ...
# User: Janet's ducks lay 16 eggs per day. ...
# Assistant: <think>
#
# ===== r1_zero_three_shot_gsm8k.prompt（1645 字符 / 9 行）
# A conversation between User and Assistant. ...
# User: There are 15 trees in the grove. ...
# Assistant: <think> There are 15 trees originally. ... <answer> 6 </answer>
# ... 省略 4 行 ...
# User: Janet's ducks lay 16 eggs per day. ...
# Assistant: <think>

## 2.3 奖励函数

模型生成回答之后，要判断它对不对。数学题的答案有很多种写法：`<answer> 1/2 </answer>`、`<answer> 0.5 </answer>`、`The answer is 0.5`。字符串比对覆盖不了，得先把答案抽出来，再判断是否等价。

这一层在 `utils/drgrpo_grader.py`，用的时候直接 import：

- `r1_zero_reward_fn(response, ground_truth)`：给 r1_zero 系列用。格式要求 `<think> </think>` 和 `<answer> </answer>` 两个标签齐全，答案取 `<answer>` 里的内容。
- `question_only_reward_fn(response, ground_truth)`：给 question_only 用。格式要求最终答案写在 `\boxed{}` 里。

两个都返回三个字段：`format_reward` 格式对不对、`answer_reward` 解析出来的答案对不对、`reward` 总分。本章不给 partial credit，所以 `reward` 永远等于 `answer_reward`，`format_reward` 只用来打日志。

判等由 `grade()` 完成（内部调 math_verify）：`1/2` 与 `0.5`、`\frac{36}{2}` 与 `18` 都算对。想看实现，在文件里搜 `def r1_zero_reward_fn` 跳过去即可。

一个容易踩的点：r1_zero 的格式判定是**字面量匹配**，`</think> <answer>` 中间必须有一个空格，写成 `</think><answer>` 会被判成没格式。

下面用七个典型输出把两个函数的判分差异跑一遍。

In [ ]:
from utils.drgrpo_grader import question_only_reward_fn, r1_zero_reward_fn

cases = [
    ("格式对 · 答案对", "<think> 16-3-4=9，9*2=18 </think> <answer> 18 </answer>"),
    ("格式对 · 答案错", "<think> 16-3-4=9，9*2=19 </think> <answer> 19 </answer>"),
    ("缺 think 标签", "<answer> 18 </answer>"),
    ("答案包在 boxed 里", "<think> 9*2=18 </think> <answer> \\boxed{18} </answer>"),
    ("只有 boxed", "\\boxed{18}"),
    ("只有一句话", "The answer is 18."),
    ("只有数字", "18"),
]

for name, response in cases:
    r1 = r1_zero_reward_fn(response, "18")
    qo = question_only_reward_fn(response, "18")
    print(f"{name}：r1_zero=({r1['format_reward']:.0f},{r1['answer_reward']:.0f})"
          f"  question_only=({qo['format_reward']:.0f},{qo['answer_reward']:.0f})")

# 格式对 · 答案对：r1_zero=(1,1)  question_only=(0,0)
# 格式对 · 答案错：r1_zero=(1,0)  question_only=(0,0)
# 缺 think 标签：r1_zero=(0,0)  question_only=(0,0)
# 答案包在 boxed 里：r1_zero=(1,1)  question_only=(1,1)
# 只有 boxed：r1_zero=(0,0)  question_only=(1,1)
# 只有一句话：r1_zero=(0,0)  question_only=(0,0)
# 只有数字：r1_zero=(0,0)  question_only=(0,0)

从这张表能读出三件事：

1. 两个 grader 各认各的标记。同一句 `<answer> 18 </answer>` 对 r1_zero 是满分，对 question_only 是 0——question_only 只认 `\boxed{}`。
2. 人看是对的、机器判 0 的样本不少：`18`、`The answer is 18.`、缺 `<think>` 的 `<answer> 18 </answer>`。要人工翻看的正是这一类，**不能只看分数就断定模型答错了**。
3. 格式对、答案错是 `(1,0)`，但 `reward` 仍是 0：格式分不给到总分里，避免模型靠格式刷分。

## 2.4 实验：在 GSM8K 上评估 base 模型

对应作业里的 Problem（prompting_baselines），5 分。要做的事是拿三种提示词各跑一遍 GSM8K 测试集，看 base 模型在每种提示词下表现成什么样。

每种提示词都按 `(format_reward, answer_reward)` 把结果分成三类数一遍：`(1,1)` 格式和答案都对、`(1,0)` 格式对答案错、`(0,0)` 格式就没解析出来。

后两类还要挑样本人工看。`(1,0)` 里有一批其实是答对了、只是解析器没认出来；`(0,0)` 里能看到模型在格式之外跑偏成了什么样——是继续编题、还是复述问题、还是干脆不写答案。

统计完再回答一个问题：只给问题够不够？zero-shot 和 few-shot 分别把模型塑造成什么样？

### 数据集

数据在 `data/gsm8k/` 下，`train.jsonl` 7473 条、`test.jsonl` 1319 条。评估只用测试集，训练集留给第 3 节的 GRPO。

每条样本两个字段，下面是 `test.jsonl` 的前两条：

```json
{
  "question": "Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
  "answer": "Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer's market.\n#### 18"
}
{
  "question": "A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?",
  "answer": "It takes 2/2=<<2/2=1>>1 bolt of white fiber\nSo the total amount of fabric is 2+1=<<2+1=3>>3 bolts of fabric\n#### 3"
}
```

`answer` 是给人看的推理过程：`<<...>>` 里是中间计算标注，`####` 后面才是最终答案。

模型要学会写出这一整段；打分时只认 `####` 后面那个数，所以读数据的时候就把前面切掉。

In [ ]:
import json

records = [json.loads(line) for line in open("data/gsm8k/test.jsonl", encoding="utf-8")]
examples = [
    {"question": r["question"], "gold": r["answer"].rsplit("####", 1)[-1].strip()}
    for r in records
]

print("条数:", len(examples))
for ex in examples[:2]:
    print(ex["gold"], "|", ex["question"][:55], "...")

# 条数: 1319
# 18 | Janet's ducks lay 16 eggs per day. She eats three for brea ...
# 3 | A robe takes 2 bolts of blue fiber and half that much w ...
#
# 脚本里把这两行包成了 load_gsm8k(path)，多一个跳过空行的判断。

### 统计

三类数完，加起来应当等于总条数。不会有 `(0,1)`：格式没解析出来时拿不到答案，answer 必然也是 0——对不上就说明漏了一种组合。

准确率按文档 3.3 的口径算：total reward 就等于 answer reward（不给 partial credit），所以答对条数除以总条数。另外顺手记一下平均生成长度和被 `max_tokens` 截断的条数。

这些统计全在 `summarize` 一个函数里。

In [ ]:
def summarize(rows):
    '''
    三分类计数 + 准确率 + 平均生成长度

    rows: list  每条形如
        {"format_reward": float, "answer_reward": float,
         "num_tokens": int, "finish_reason": "stop" | "length"}
    '''
    num_fmt1_ans1 = sum(r["format_reward"] == 1 and r["answer_reward"] == 1 for r in rows)
    num_fmt1_ans0 = sum(r["format_reward"] == 1 and r["answer_reward"] == 0 for r in rows)
    num_fmt0_ans0 = sum(r["format_reward"] == 0 and r["answer_reward"] == 0 for r in rows)
    return {
        "num_fmt1_ans1": num_fmt1_ans1,
        "num_fmt1_ans0": num_fmt1_ans0,
        "num_fmt0_ans0": num_fmt0_ans0,
        "accuracy": num_fmt1_ans1 / len(rows) if rows else 0.0,
        "average_num_tokens": sum(r["num_tokens"] for r in rows) / len(rows) if rows else 0.0,
        "num_truncated": sum(r["finish_reason"] == "length" for r in rows),
    }

In [ ]:
fake = [
    {"format_reward": 1.0, "answer_reward": 1.0, "num_tokens": 120, "finish_reason": "stop"},
    {"format_reward": 1.0, "answer_reward": 0.0, "num_tokens": 512, "finish_reason": "length"},
    {"format_reward": 0.0, "answer_reward": 0.0, "num_tokens": 40,  "finish_reason": "stop"},
]
print(summarize(fake))

# {'num_fmt1_ans1': 1, 'num_fmt1_ans0': 1, 'num_fmt0_ans0': 1,
#  'accuracy': 0.3333333333333333, 'average_num_tokens': 224.0, 'num_truncated': 1}

# 三类加起来等于总条数。对不上就说明漏了一种组合